## Query Orders Data as JSON Strings

1. Extract Top Level Column Values
2. Extract Array elements
3. Extract Nested Column Values
4. CAST Column Values to a Specific Data Type


In [0]:
%sql
SELECT * FROM gizmobox.bronze.v_orders

### Extract Top Level Column Values

In [0]:
%sql
SELECT value:order_id FROM gizmobox.bronze.v_orders

### Extract Array elements

In [0]:
%sql
SELECT value:items FROM gizmobox.bronze.v_orders

### CAST Column Values to a Specific Data Type

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW tv_orders_fixed AS
SELECT
    regexp_replace(
        value,
        '"order_date"\\s*:\\s*(\\d{4}-\\d{2}-\\d{2})',
        '"order_date":"$1"'
    ) AS fixed_value
FROM gizmobox.bronze.v_orders;

In [0]:
%sql
SELECT * FROM tv_orders_fixed

In [0]:
%sql
SELECT schema_of_json(fixed_value) as json_schema FROM tv_orders_fixed
LIMIT 1

In [0]:
%sql
SELECT from_json(
    fixed_value,
    "STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>"
) as json_value FROM tv_orders_fixed

In [0]:
%sql
CREATE TABLE gizmobox.silver.orders_json AS
SELECT from_json(
    fixed_value,
    "STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>"
) as json_value FROM tv_orders_fixed

In [0]:
%sql
SELECT * FROM gizmobox.silver.orders_json